In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [ ]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "data" / "anime_cache.json")

In [4]:
anime_data = anime_data_client.get_cache()

Build features

In [5]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df = builder.build_features()

builder.svd_explained_variance

np.float64(0.37937105892729933)

Convert each anime in df to vectors

In [6]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [7]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

## Unclipped Bayesian Score Check

This sweep tests ranking with raw, unclipped Bayesian predictions. The goal is to check whether preserving out-of-range score ordering improves recommendations compared with clipping predictions back to the 1-10 MAL rating range before applying the uncertainty penalty.

In [8]:
score_variant = "unclipped"
clip_predictions = False
weights_uncertainty = [14]
n_runs = 200
tuning_top_ks = [5, 10]

from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator

hitman = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

(
    bayesian_results,
    bayesian_summary,
    best_bayesian_weights,
    baseline_results,
    baseline_summary,
) = hitman.tune_bayesian_uncertainty(
    weights=weights_uncertainty,
    n_runs=n_runs,
    top_ks=tuning_top_ks,
    random_state=42,
    clip_predictions=clip_predictions,
)

ranking_evaluator = RankingMetricEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

ranking_results, ranking_summary = ranking_evaluator.tune_bayesian_uncertainty_ranking(
    weights=weights_uncertainty,
    n_runs=n_runs,
    top_ks=tuning_top_ks,
    random_state=42,
    clip_predictions=clip_predictions,
)

bayesian_summary = bayesian_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)
ranking_summary = ranking_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

unclipped_average_metrics = (
    bayesian_summary
    .merge(ranking_summary, on=["bayesian_uncertainty_weight", "k"], how="left")
    .merge(baseline_summary, on="k", how="left")
)
unclipped_average_metrics.insert(0, "score_variant", score_variant)
unclipped_average_metrics.insert(1, "clip_predictions", clip_predictions)
unclipped_average_metrics["n_runs"] = n_runs

display(unclipped_average_metrics)


,score_variant,clip_predictions,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,avg_ndcg_at_k,...,avg_relevant_hits_at_k,avg_strong_hits_at_k,avg_test_relevant,avg_test_strong_relevant,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,n_runs
0,unclipped,False,14,5,0.5520,0.209560,0.086250,0.032744,2.760,0.665909,...,3.365,2.745,46.235,31.76,0.1150,0.130615,0.017969,0.020409,0.575,200
1,unclipped,False,14,10,0.3945,0.137529,0.123281,0.042978,3.945,0.515066,...,4.960,3.700,46.235,31.76,0.0575,0.065308,0.017969,0.020409,0.575,200


Clipped

In [9]:
score_variant = "clipped"
clip_predictions = True
weights_uncertainty = [14]
n_runs = 200
tuning_top_ks = [5, 10]

from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator

hitman = HitRateEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

(
    bayesian_results,
    bayesian_summary,
    best_bayesian_weights,
    baseline_results,
    baseline_summary,
) = hitman.tune_bayesian_uncertainty(
    weights=weights_uncertainty,
    n_runs=n_runs,
    top_ks=tuning_top_ks,
    random_state=42,
    clip_predictions=clip_predictions,
)

ranking_evaluator = RankingMetricEvaluator(
    anime_df_scaled=anime_df_scaled,
    anime_df=anime_df,
    scores=user_scores,
    anime_data_client=anime_data_client,
    anime_data=anime_data,
    builder=builder,
    recommender=recommender,
)

ranking_results, ranking_summary = ranking_evaluator.tune_bayesian_uncertainty_ranking(
    weights=weights_uncertainty,
    n_runs=n_runs,
    top_ks=tuning_top_ks,
    random_state=42,
    clip_predictions=clip_predictions,
)

bayesian_summary = bayesian_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)
ranking_summary = ranking_summary.rename(
    columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
)

clipped_average_metrics = (
    bayesian_summary
    .merge(ranking_summary, on=["bayesian_uncertainty_weight", "k"], how="left")
    .merge(baseline_summary, on="k", how="left")
)
clipped_average_metrics.insert(0, "score_variant", score_variant)
clipped_average_metrics.insert(1, "clip_predictions", clip_predictions)
clipped_average_metrics["n_runs"] = n_runs

comparison_metrics = clipped_average_metrics.copy()
if "unclipped_average_metrics" in globals():
    comparison_metrics = pd.concat(
        [unclipped_average_metrics, clipped_average_metrics],
        ignore_index=True,
    )

metrics_path = (
    PROJECT_ROOT
    / "metrics"
    / "current_corpus_4793_anime"
    / "clipped_unclipped_bayesian_200run_ndcg_2026.csv"
)
metrics_path.parent.mkdir(parents=True, exist_ok=True)
comparison_metrics.to_csv(metrics_path, index=False)

print(f"Saved {metrics_path.relative_to(PROJECT_ROOT)}")
display(comparison_metrics)


Saved metrics\current_corpus_4793_anime\clipped_unclipped_bayesian_200run_ndcg_2026.csv


,score_variant,clip_predictions,bayesian_uncertainty_weight,k,avg_precision_at_k,std_precision_at_k,avg_hit_rate,std_hit_rate,avg_hits,avg_ndcg_at_k,...,avg_relevant_hits_at_k,avg_strong_hits_at_k,avg_test_relevant,avg_test_strong_relevant,baseline_avg_precision_at_k,baseline_std_precision_at_k,baseline_avg_hit_rate,baseline_std_hit_rate,baseline_avg_hits,n_runs
0,unclipped,False,14,5,0.5520,0.209560,0.086250,0.032744,2.760,0.665909,...,3.365,2.745,46.235,31.76,0.1150,0.130615,0.017969,0.020409,0.575,200
1,unclipped,False,14,10,0.3945,0.137529,0.123281,0.042978,3.945,0.515066,...,4.960,3.700,46.235,31.76,0.0575,0.065308,0.017969,0.020409,0.575,200
2,clipped,True,14,5,0.5490,0.210286,0.085781,0.032857,2.745,0.659567,...,3.335,2.705,46.235,31.76,0.1150,0.130615,0.017969,0.020409,0.575,200
3,clipped,True,14,10,0.3925,0.137434,0.122656,0.042948,3.925,0.509294,...,4.905,3.635,46.235,31.76,0.0575,0.065308,0.017969,0.020409,0.575,200


## Results

Rerun the unclipped and clipped cells above to produce the current comparison. Both cells now use `n_runs=200`, `bayesian_uncertainty_weight=14`, `top_ks=[5, 10]`, and `random_state=42`. Each variant reports the hit-rate metrics (`avg_precision_at_k`, `avg_hit_rate`, `avg_hits`) plus ranking metrics from `RankingMetricEvaluator` (`avg_ndcg_at_k`, `avg_mrr_at_k`, relevant hits, and strong hits).

The clipped cell combines both variants when `unclipped_average_metrics` is already in memory and saves the summary to `metrics/current_corpus_4793_anime/clipped_unclipped_bayesian_200run_ndcg_2026.csv`.

Decision rule: if precision remains close, prefer the variant with better NDCG/MRR and more faithful ranking behavior. My current preference is still to keep Bayesian ranking **unclipped** unless the 200-run NDCG results show a clear clipped advantage, because raw scores preserve the model's ordering instead of flattening out-of-range predictions.
